# 3.1b — dilated ConvNeXt at 128px, on an A100

**One arm.** `v31-convnext_dilated_128`, split out of `31_dilation.ipynb` so it can run
alone.

## Why this arm

The v31 control settled the question on `dilated_style` at 64px, and the answer was the
strongest confirmed effect in the project after augmentation:

| | rates | receptive field | macro-F1 | 95% CI |
|---|---|---|---|---|
| `v31-dilated_rotation_64` | 1,2,4,8 x2 | 63x63 | **0.8852** | [0.8714, 0.8955] |
| `v31-dilated_control_64` | all 1 | 19x19 | 0.8216 | [0.8065, 0.8347] |

**+0.0636 with non-overlapping intervals**, at identical parameter count. The gain was
`Loc` (0.621 → 0.754), not `Scratch` — a diffuse cluster needs context to separate from
`Edge-Loc`, and that is exactly what a 63x63 field buys.

This asks whether it transfers to an architecture that **already downsamples**.
`v28-convnext_big_128` is the same model without dilation at **0.8988**, so the pair is
dilation alone — the rates cost no parameters.

## Speed

The earlier attempt measured **522 s/epoch on a T4**. An A100 is roughly 3-5x a T4 on fp16
tensor-core work, so expect **~110-170 s/epoch**, putting 60 epochs at **2-3 h** — and early
stopping will probably end it near 40.

Section 4 measures this directly before committing to it, so you are not guessing.

It is not the dataloader. At 128px the geometry cache fits (121,063 x 128² = 1.98 GB, under
the 2 GiB budget) and `transform_device=cuda` moves the one-hot and rotation onto the GPU.
The cost is the model.

What the tuning cell does:

* **TF32 on** — Ampere only, so it does nothing on a T4 and something here. Training already
  runs fp16 autocast for the heavy matmuls, so this touches only what autocast leaves in
  fp32: real but modest.
* **`prefetch_factor` 4** — smooths loader jitter; cannot fix a throughput deficit.
* **`max_epochs` 60 instead of 100** — the largest saving, and free: every comparable arm
  found its best well inside 60 (`v28-convnext_big_128` at 30 of 40,
  `v31-dilated_rotation_64` at 23 of 33). Patience 10 still decides.

**Batch stays at 256.** An A100 could hold 512 and it would be ~1.4x faster, but the learning
rate is square-root scaled to 256 and the control ran at 256; changing it would confound
dilation with batch size. `DOUBLE_BATCH` is there if you decide the time is worth it.

### `channels_last`, the one real lever

Dilated **depthwise** convolutions can miss cuDNN's fast path — on the T4, dilation cost ~47%
over the undilated model (522 vs 356 s/epoch). NHWC is the layout those fast kernels actually
want, and it is where the NCHW path is worst.

`trainer.channels_last` is a new flag, off by default everywhere else, so no other config
changes. It is a **memory-layout** change, not an arithmetic one: outputs and gradients match
NCHW to 1e-5, and `tests/test_channels_last.py` asserts both.

Section 4 times three variants — undilated, dilated, and dilated in `channels_last` — and sets
`USE_CHANNELS_LAST` from what actually won on this GPU. Section 5 reads it. No guessing.

## 0. Colab web UI only — clone and authenticate

Skip if `/content/fdl-project` already exists.

In [ ]:
from getpass import getpass
from pathlib import Path
import subprocess

TARGET = Path("/content/fdl-project")
BRANCH = "feature/phase3-architectures"
REMOTE = "github.com/ezero3/fdl-project.git"


def run(*command: str) -> None:
    subprocess.run(command, check=True)


if TARGET.exists():
    print(f"{TARGET} already present -- pulling")
    run("git", "-C", str(TARGET), "fetch", "origin", BRANCH)
    run("git", "-C", str(TARGET), "checkout", BRANCH)
    run("git", "-C", str(TARGET), "pull", "--ff-only")
else:
    # Private repo, so the clone needs a personal access token. getpass keeps it
    # out of the notebook and out of the output.
    token = getpass("GitHub personal access token (input hidden): ").strip()
    run("git", "clone", "--branch", BRANCH,
        f"https://{token}@{REMOTE}", str(TARGET))
    # Drop the token from the stored remote; a later pull will ask again rather
    # than leaving a credential sitting in .git/config.
    run("git", "-C", str(TARGET), "remote", "set-url", "origin", f"https://{REMOTE}")
    del token

print(subprocess.run(["git", "-C", str(TARGET), "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout.strip())

## 1. Setup

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path


def looks_like_the_repository(path: Path) -> bool:
    return (path / "pyproject.toml").exists() and (path / "src" / "fdl_project").is_dir()


REPO = next(
    (p for p in [Path.cwd(), *Path.cwd().parents, Path("/content/fdl-project")]
     if looks_like_the_repository(p)),
    None,
)
assert REPO is not None, "clone the repo to /content/fdl-project first"
os.chdir(REPO)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--ignore-requires-python",
                "-e", str(REPO), "--no-deps"], check=True)
source = str(REPO / "src")
if source not in sys.path:
    sys.path.insert(0, source)

import torch

DRIVE_ROOT = Path("/content/drive/MyDrive")
DRIVE = DRIVE_ROOT / "BICOCCA/FDL"
DATASET = REPO / "data/MIR-WM811K/WM811K.pkl"
EXPECTED_BYTES = 2_022_961_642


def mount_drive() -> bool:
    if DRIVE_ROOT.is_dir():
        return True
    try:
        from google.colab import drive

        drive.mount("/content/drive")   # idempotent; never force_remount
    except Exception as error:
        print(f"  Drive unavailable ({type(error).__name__})")
        return False
    return DRIVE_ROOT.is_dir()


HAS_DRIVE = mount_drive()
if not DATASET.exists():
    DATASET.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE / "DATA/data/MIR-WM811K/WM811K.pkl", DATASET)
assert DATASET.stat().st_size == EXPECTED_BYTES, "wrong pickle: splits are row indices"

CHECKPOINTS = DRIVE / "checkpoints"
if HAS_DRIVE:
    CHECKPOINTS.mkdir(parents=True, exist_ok=True)

print(f"gpu     {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"drive   {'mounted' if HAS_DRIVE else 'NOT mounted'}")
print(f"dataset {DATASET.stat().st_size / 1024**3:.2f} GiB")

## 2. W&B

In [ ]:
USE_WANDB = True
WANDB_PROJECT = "wm811k-wafer-defects"

if USE_WANDB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)
    import wandb

    if not wandb.api.api_key:
        # Colab Secrets work on the web UI. They time out under the VS Code
        # runtime, which is why the other notebooks tell you to use a terminal.
        # Add WANDB_KEY at the key icon in the left sidebar and enable it here.
        try:
            from google.colab import userdata

            wandb.login(key=userdata.get("WANDB_KEY"))
        except Exception as error:
            print(f"  Colab Secrets unavailable ({type(error).__name__})")

    if not wandb.api.api_key:
        USE_WANDB = False
        print("  not authenticated -- add WANDB_KEY to Colab Secrets, or run "
              "`wandb login` in a terminal, then rerun this cell")
    else:
        print(f"  wandb ready, project {WANDB_PROJECT!r}")


## 3. A100 tuning

Run before section 4 — these are process-wide flags and have to be set before CUDA does
any real work.

In [ ]:
"""A100 tuning. Everything here is either free or explicitly reversible."""

import os

import torch

# TF32: A100 tensor cores run fp32 matmuls at ~8x throughput with 10-bit
# mantissas. Training already uses fp16 autocast for the heavy matmuls, so this
# only touches what autocast leaves in fp32 -- real but modest. It changes
# numerics slightly; far below the 0.02 noise floor.
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
# Already set by seed_everything(deterministic=False), repeated here so the
# state is visible rather than assumed.
torch.backends.cudnn.benchmark = True

name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"
memory = (torch.cuda.get_device_properties(0).total_memory / 1024**3
          if torch.cuda.is_available() else 0)
print(f"gpu      {name}  {memory:.0f} GiB")
print(f"vcpu     {os.cpu_count()}")
print(f"tf32     matmul={torch.backends.cuda.matmul.allow_tf32} "
      f"cudnn={torch.backends.cudnn.allow_tf32}")
if "A100" not in name:
    print("  NOTE: TF32 only helps on Ampere or newer. This is not an A100.")

## 4. Benchmark before committing

Thirty seconds of measurement instead of an hour of guessing. Times the real model at the
real batch and resolution, dilated and undilated, and projects the epoch cost.

The ratio is the number to read: on a T4 dilation cost ~1.47x, and whether that penalty
follows to Ampere decides whether this arm is worth its hours.

In [ ]:
"""Thirty seconds of measurement instead of an hour of guessing.

Times one forward+backward step of the real model at the real batch size and
resolution, dilated and undilated, and projects the epoch cost from it. The
ratio between the two is the dilation penalty on THIS GPU -- on a T4 it was
about 1.47x, and whether that follows to Ampere is the open question.
"""

import time

import torch

from fdl_project.config.loader import load_experiment_config
from fdl_project.config.registry import build_model

TRAIN_SAMPLES = 121_063
BATCH = 256

probe_config = load_experiment_config(
    REPO / "configs/train/v31_dilated/13_convnext_dilated_128.yaml"
)
base_kwargs = dict(probe_config.model.kwargs)
size = probe_config.data.preprocessing.target_size[0]


def time_one_step(kwargs: dict, steps: int = 6, channels_last: bool = False) -> float:
    model = build_model(probe_config.model.name, **kwargs).cuda().train()
    if channels_last:
        model = model.to(memory_format=torch.channels_last)
    optimizer = torch.optim.AdamW(model.parameters(), lr=7e-4)
    scaler = torch.amp.GradScaler("cuda")
    inputs = torch.rand(BATCH, 3, size, size, device="cuda")
    if channels_last:
        inputs = inputs.contiguous(memory_format=torch.channels_last)
    targets = torch.randint(0, 9, (BATCH,), device="cuda")

    for index in range(steps + 2):          # first two warm up cudnn.benchmark
        if index == 2:
            torch.cuda.synchronize()
            started = time.perf_counter()
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda"):
            loss = torch.nn.functional.cross_entropy(model(inputs), targets)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
    torch.cuda.synchronize()
    elapsed = (time.perf_counter() - started) / steps
    del model, optimizer, inputs, targets
    torch.cuda.empty_cache()
    return elapsed


undilated = time_one_step({**base_kwargs, "dilation": 1})
dilated = time_one_step(base_kwargs)
nhwc = time_one_step(base_kwargs, channels_last=True)
steps_per_epoch = TRAIN_SAMPLES / BATCH

print(f"batch {BATCH} at {size}px, forward+backward only (no dataloader):")
for label, seconds in [("no dilation, NCHW", undilated),
                       ("dilated, NCHW", dilated),
                       ("dilated, channels_last", nhwc)]:
    print(f"  {label:24} {seconds*1000:7.1f} ms/step"
          f"  ~= {seconds*steps_per_epoch:6.0f} s/epoch")

print(f"\n  dilation penalty:      {dilated/undilated:.2f}x   (~1.47x on a T4)")
print(f"  channels_last speedup: {dilated/nhwc:.2f}x")

# Use the layout that actually won here, rather than assuming.
USE_CHANNELS_LAST = nhwc < dilated * 0.97
best = min(dilated, nhwc)
projected_hours = best * steps_per_epoch * 60 / 3600
print(f"\nUSE_CHANNELS_LAST = {USE_CHANNELS_LAST}  (section 5 reads this)")
print(f"60 epochs, compute only: ~{projected_hours:.1f} h "
      f"(validation and the loader add on top)")
if projected_hours > 5:
    print("  Too long. Drop this arm, or set DOUBLE_BATCH in section 5.")
else:
    print("  Workable -- go on to section 5.")

## 5. Train

`MAX_EPOCHS` and `DOUBLE_BATCH` are at the top of the cell. Checkpoints go to Drive under
this arm's name, so a dropped runtime resumes rather than restarts.

In [ ]:
import shutil, time

import pandas as pd

from fdl_project.config.loader import load_experiment_config
from fdl_project.config.registry import build_model
from fdl_project.data.datasets import load_wm811k_dataframe
from fdl_project.models.baseline_cnn import count_trainable_parameters
from fdl_project.training.runner import run_experiment

SERIES = "v31_dilated"
CONFIG = REPO / "configs/train" / SERIES / "13_convnext_dilated_128.yaml"
assert CONFIG.exists(), f"{CONFIG} missing -- git pull, then Runtime > Restart session"

# 60, not the config's 100. Every comparable arm found its best well inside it:
# v28-convnext_big_128 at 30 of 40, v31-dilated_rotation_64 at 23 of 33,
# v28-convnext_big_64 at 25 of 35. Patience 10 still decides; this only caps the
# worst case, and it is the single biggest time saving available here.
MAX_EPOCHS = 60

# Batch stays at 256. An A100 could hold 512 and it would be roughly 1.4x
# faster, but the learning rate is square-root scaled to 256 and this arm is
# measured against v28-convnext_big_128, which ran at 256. Changing it would
# confound dilation with batch size. Set this True only if you accept that.
DOUBLE_BATCH = False

OUTPUT = REPO / "output" / SERIES
OUTPUT.mkdir(parents=True, exist_ok=True)

# Set by the benchmark in section 4; default False if that cell was skipped.
USE_CHANNELS_LAST = globals().get("USE_CHANNELS_LAST", False)

OVERRIDES = ["data.transform_device=cuda", f"trainer.max_epochs={MAX_EPOCHS}",
             "data.prefetch_factor=4",
             f"trainer.channels_last={str(USE_CHANNELS_LAST).lower()}"]
if DOUBLE_BATCH:
    OVERRIDES += ["trainer.batch_size=512", "optimizer.kwargs.lr=1.0e-3"]
if HAS_DRIVE:
    OVERRIDES.append(f"checkpoint.directory={CHECKPOINTS}")
if USE_WANDB:
    OVERRIDES += ["logging.wandb.enabled=true",
                  f"logging.wandb.project={WANDB_PROJECT}",
                  f"logging.wandb.tags=[{SERIES},dilation]"]

config = load_experiment_config(CONFIG, overrides=OVERRIDES)
assert config.model.name == "convnext_style", "this notebook runs one arm"
assert config.data.augmentation.name == "rotation", "settled pipeline is rotation"

model = build_model(config.model.name, **config.model.kwargs)
print(f"=== {config.name}")
print(f"    {count_trainable_parameters(model):,} parameters, dilation "
      f"{model.dilation_rates}, {config.data.preprocessing.target_size[0]}px")
print(f"    batch {config.trainer.batch_size}, lr {config.optimizer.kwargs['lr']:g}, "
      f"max_epochs {config.trainer.max_epochs}, patience "
      f"{config.trainer.early_stopping.patience}")
print(f"    channels_last {config.trainer.channels_last}")
del model

dataframe = load_wm811k_dataframe(DATASET)
started = time.monotonic()
result = run_experiment(config, overwrite=True, dataframe=dataframe)

macro = result.bootstrap.aggregate.set_index("metric").loc["macro_f1"]
per_class = result.validation.per_class_metrics.set_index("class_name")["f1"]
history = result.fit.history
row = {
    "run": config.name,
    "dilation": str(config.model.kwargs["dilation"]),
    "px": config.data.preprocessing.target_size[0],
    "macro_f1": round(float(macro.point_estimate), 4),
    "ci_lower": round(float(macro.ci_lower), 4),
    "ci_upper": round(float(macro.ci_upper), 4),
    "scratch_f1": round(float(per_class["Scratch"]), 3),
    "loc_f1": round(float(per_class["Loc"]), 3),
    "best_epoch": result.fit.best_epoch,
    "epochs": len(history),
    "s_per_epoch": round(float(history["epoch_seconds"].median()), 1),
    "minutes": round((time.monotonic() - started) / 60, 1),
}
csv = OUTPUT / "convnext_dilated_128_results.csv"
pd.DataFrame([row]).to_csv(csv, index=False)
if HAS_DRIVE:
    shutil.copy2(csv, DRIVE / f"{SERIES}_convnext_dilated_128.csv")

print(f"\n  macro-F1 {row['macro_f1']:.4f} [{row['ci_lower']:.4f}, {row['ci_upper']:.4f}]")
print(f"  Scratch {row['scratch_f1']:.3f}   Loc {row['loc_f1']:.3f}")
print(f"  best epoch {row['best_epoch']}/{row['epochs']}   "
      f"{row['s_per_epoch']:.0f} s/epoch   {row['minutes']:.1f} min")
if row["best_epoch"] >= row["epochs"] - 2:
    print("  STILL IMPROVING AT THE CAP -- this number is a floor.")
print(f"  saved to {csv}")

## 6. Where it lands

Run after section 4 finishes.

In [ ]:
# The pairing this arm exists for: the identical model without dilation.
CONTROL = ("v28-convnext_big_128", 0.8988, 0.787)   # macro_f1, Scratch
NOISE_FLOOR = 0.02

MEASURED = [
    ("v32-resnet34_finetune",       "pretrained CNN 21.4M, 128px", 0.9041, 0.793),
    ("v32-vit_b_32_finetune",       "pretrained ViT 87.7M, 224px", 0.9029, None),
    ("dilated-style-64-dihedral8",  "ours 298k, dihedral8",        0.8995, 0.793),
    ("v28-convnext_big_128",        "ours 2.68M, 128px, NO dil",   0.8988, 0.787),
    ("v30-finetune_encoder_1e-5",   "pretrained CNN 11.3M",        0.8938, 0.813),
    ("v33-efficientnet_b0_finetune","pretrained CNN 4.3M",         0.8923, None),
    ("v27-resnet_style",            "ours 2.83M, 64px",            0.8900, 0.759),
    ("v27-convnext_style",          "ours 414k, 64px",             0.8883, 0.736),
    ("v28-resnet_style_128",        "ours 2.83M, 128px",           0.8878, 0.800),
    ("v31-dilated_rotation_64",     "ours 298k, DILATED",          0.8852, 0.705),
    ("v28-vit_style_128",           "ours ViT 2.83M",              0.8744, None),
    ("v27-baseline_cnn",            "ours 157k, 64px",             0.8646, 0.715),
    ("v31-dilated_control_64",      "ours 298k, dilation OFF",     0.8216, 0.722),
]

table = pd.DataFrame(MEASURED, columns=["run", "what", "macro_f1", "scratch_f1"])
mine = pd.DataFrame([{"run": row["run"], "what": "THIS NOTEBOOK (dilated convnext)",
                      "macro_f1": row["macro_f1"], "scratch_f1": row["scratch_f1"]}])
pd.set_option("display.width", 215)
display(pd.concat([mine, table]).sort_values("macro_f1", ascending=False).reset_index(drop=True))

delta = row["macro_f1"] - CONTROL[1]
verdict = "REAL" if abs(delta) > NOISE_FLOOR else "inside the noise floor"
print(f"Dilation on convnext_big at 128px: {delta:+.4f} vs {CONTROL[0]} "
      f"({CONTROL[1]:.4f})  [{verdict}]")
print(f"  Scratch {row['scratch_f1']:.3f} against {CONTROL[2]:.3f}")
print(f"  Loc     {row['loc_f1']:.3f}")
print()
print("On dilated_style at 64px the same switch was worth +0.0636 with")
print("non-overlapping intervals (0.8852 vs 0.8216), and the gain was Loc")
print("(0.621 -> 0.754), not Scratch. Whether that transfers to an")
print("architecture that already downsamples is what this arm answers.")

## 7. Report back

```
convnext_dilated_128: macro-F1 X.XXXX [lo, hi], Scratch X.XXX, Loc X.XXX, N s/epoch
```

The number that decides whether dilation is a project-wide finding or a `dilated_style`
quirk is the delta against `v28-convnext_big_128` (0.8988).

CSV at `output/v31_dilated/convnext_dilated_128_results.csv`, copied to Drive.